<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-5-Lab-1/Unit_5_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import entropy
from ipywidgets import Dropdown, IntSlider, Checkbox, Button, VBox, HBox, Output
from IPython.display import display

# --- Load dataset ---
url = "https://raw.githubusercontent.com/azcsprof/ASU-CSE475-SS25/Unit-5-Lab-1/Customer_data.csv"
df = pd.read_csv(url)

# --- Clean dataset ---
if df.shape[1] == 1 and df.iloc[0, 0].count(",") >= 2:
    df = df.iloc[:, 0].str.split(",", expand=True)

df.columns = df.columns.astype(str).str.strip()
if all(col.isdigit() for col in df.columns):
    df.columns = ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1–100)'][:df.shape[1]]

if 'Gender' in df.columns and not pd.api.types.is_numeric_dtype(df['Gender']):
    df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(df['Gender'])

# Convert all columns to numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop ID column if present
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in ['CustomerID', 'ID', 'customer_id']:
    if col in numeric_cols:
        numeric_cols.remove(col)

# Remove 'Gender' from centroid stats, but keep it for display
centroid_cols = [col for col in numeric_cols if col != 'Gender']

# --- Widgets ---
method_dropdown = Dropdown(options=['KMeans', 'Hierarchical'], value='KMeans', description='Method')
k_slider = IntSlider(min=2, max=10, value=3, description='Clusters')
dendro_checkbox = Checkbox(value=False, description='Show Dendrogram')
run_button = Button(description="Run Clustering", button_style='success')
output_area = Output()

def compute_entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 0.0 if len(counts) <= 1 else entropy(probs, base=2) / np.log2(len(counts))

def on_run_clicked(b):
    output_area.clear_output()
    with output_area:
        k = k_slider.value
        method = method_dropdown.value
        show_dendro = dendro_checkbox.value

        data = df[centroid_cols].dropna()
        scaler = StandardScaler()
        X = scaler.fit_transform(data)

        if method == 'Hierarchical':
            Z = linkage(X, method='ward')
            if show_dendro:
                plt.figure(figsize=(10, 5))
                dendrogram(Z, truncate_mode='lastp', p=30)
                plt.title("Dendrogram")
                plt.tight_layout()
                plt.show()
            labels = fcluster(Z, k, criterion='maxclust') - 1
        else:
            model = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = model.fit_predict(X)
            centroids = pd.DataFrame(
                scaler.inverse_transform(model.cluster_centers_),
                columns=centroid_cols
            )
            print("📍 Centroid Values (Original Scale):")
            display(centroids)

        # Plot PCA projection
        X_2d = PCA(n_components=2).fit_transform(X)
        plt.figure(figsize=(6, 5))
        palette = sns.color_palette("Set2", len(np.unique(labels)))
        for i in np.unique(labels):
            plt.scatter(X_2d[labels == i, 0], X_2d[labels == i, 1], label=f"Cluster {i}", s=50)
        plt.xlabel("PCA 1")
        plt.ylabel("PCA 2")
        plt.title(f"{method} Clustering (k={k})")
        plt.legend()
        plt.tight_layout()
        plt.show()

        print(f"✅ Silhouette Score: {silhouette_score(X, labels):.3f}")
        print(f"🧮 Cluster Size Entropy: {compute_entropy(labels):.3f} (0=imbalanced, 1=even)")

        print("\n📏 Cluster Sizes:")
        display(pd.Series(labels).value_counts().sort_index())

        # Cluster stats (exclude ID, include Gender only for context)
        result_df = df.loc[data.index].copy()
        result_df['Cluster'] = labels
        print("\n📋 Descriptive Stats per Cluster:")
        stats = result_df.groupby('Cluster')[numeric_cols].agg(['mean', 'std', 'count'])
        display(stats)

run_button.on_click(on_run_clicked)

controls = VBox([
    HBox([method_dropdown, k_slider, dendro_checkbox]),
    run_button,
    output_area
])
display(controls)

## Interpreting Clustering Results: A Student Guide

This section will help you make sense of the clustering output. Use it to reflect on what each cluster reveals about your data.

---

### Cluster Plot (PCA View)

A scatterplot using PCA (Principal Component Analysis) helps you **visually assess** the cluster separation.

- Compact blobs of color → clean clusters  
- Overlapping points → clustering may be unclear  
- Use this to **confirm or question** what the metrics suggest

### Evaluation Metrics

- **Silhouette Score**  
  Indicates how well each point fits within its assigned cluster.  
  - Values near **1.0** = well-defined clusters  
  - Values near **0.0** = overlapping or ambiguous clusters  
  - Values below **0.0** = points may be in the wrong cluster  

- **Cluster Size Entropy**  
  Measures how evenly clusters are populated.  
  - **1.0** = all clusters have similar sizes  
  - **0.0** = one or more clusters dominate  

---

### Cluster Size Summary

Check how many data points ended up in each cluster.  
- Large imbalances may suggest:  
  - Poor feature scaling  
  - Uneven natural groupings  
  - Need for different clustering method  

---

### Descriptive Stats by Cluster

Each cluster includes the **mean**, **standard deviation**, and **count** for all numeric features.

- Use **means** to identify key traits (e.g., age, income, gender distribution)
- Use **std** to gauge how similar members of the cluster are
- Ask:
  - What type of person does each cluster represent?
  - Which cluster might be a marketing target?
  - Do any clusters show surprising combinations (e.g., high income + low spending)?

---

### Dendrogram (if shown)

- Shows **hierarchical relationships** between observations  
- A horizontal cut through the dendrogram suggests a possible cluster count  
- Use to compare with KMeans or DBSCAN results  

---

### Questions to Ask

- What patterns does each cluster represent?
- Do the results match real-world expectations?
- Are certain clusters worth exploring further?
- Would different features or a different number of clusters change the story?

---